# Ejemplo: combinatoria

:::{seealso} Revisa la física
Puedes revisar la teoría en un libro de física estadística, como @Stowe2007.
:::

> Considera 50 partículas elementales de espín 1/2
> (distinguibles y sin campos externos presentes).
> ¿Cuál es la probabilidad de que 40 de ellas tengan espín positivo y el resto tengan
> espín negativo? ¿Cuántas configuraciones pueden dar este resultado?
>
> -- *Stowe, capítulo 2*

Si una partícula tiene una probabilidad $p$ de cumplir un criterio,
la probabilidad de que $n$ de $N$ partículas la cumplan es
\begin{align*}
  {P_N}(n) &= \frac{N!}{n! (N - n)!} p^n (1 - p)^{N - n} \\
    &= \binom{N}{n} p^n (1 - p)^{N - n}.
\end{align*}
El coeficiente $ \binom{N}{n} \equiv {N!}/{n! (N - n)!} $ es llamado el coeficiente
binomial, y está implementado en el módulo `math`.

In [ ]:
import math

Implementemos la fórmula general en una función.

In [ ]:
def probability_onecrit(N: int, n: int, p: float) -> float:
    "Probabilidad de que n de N objetos cumplan un criterio de probabilidad p."
    # ^ Esto es un *docstring*, texto que documenta la función.

    if not (0.0 <= p <= 1.0):
        raise ValueError("La probabilidad debe ser no negativa, menor o igual a 1.")

    P = math.comb(N, n) * (p**n) * ((1.0 - p) ** (N - n))
    return P

Este pequeño fragmento de código nos permite incorporar muchas lecciones de inmediato.

::: {tip} Lección de programación
Escribe código reutilizable dentro de funciones. Esto hace al código más fácil de leer,
de mantener, y refleja la forma en la que pensamos al hacer trabajo científico.
:::

::: {tip} Lección de programación
Documenta tu código. La manera más simple es añadiendo *docstrings* a tus funciones.
:::

In [ ]:
help(probability_onecrit)

::: {tip} Lección de programación
Incorpora métodos para asegurarte de que tus funciones se usan correctamente.
En contextos científicos es extremadamente importante de que los programas hagan lo
que creemos que están haciendo. Asegúrate de atrapar errores que puedas cometer tú o
la gente que use tu código en el futuro.
:::

::: {tip} Lección de programación
Usa sugerencias de tipo. Si una cantidad debe ser entera, asignarle un número con decimal
es una señal obvia de que algo salió mal.
:::

Fijemos los datos del problema y calculemos lo que nos pide el problema.

In [ ]:
particle_number: int = 50
up_number: int = 40
up_probability: float = 0.5

print(f"La probabilidad es {
  probability_onecrit(particle_number, up_number, up_probability):.3e}")

print(
    f"Existen {math.comb(particle_number, up_number):.3e} combinaciones",
    f"para {up_number} de {particle_number} partículas.",
)

> Se lanzan 21 dados.
> ¿Cuál es la probabilidad de que uno de ellos caiga en 1, dos en 2, tres en 3… y
> seis en 6?

Para $N$ partículas que caen en una de $m$ categorías, donde la probabilidad de
caer en la $j$-ésima categoría es $p_j$, la probabilidad de que
$n_1$ estén en la primera, $n_2$ en la segunda, etc. es
\begin{align*}
  {P_N}(n_1, n_2, \ldots, n_m)
    &= \binom{N}{n_1, n_2, \ldots, n_m} \prod_{j = 1}^{m} p_{j}^{n_j} \\
    &= \frac{N!}{\prod_{k = 1}^{m} n_k!} \prod_{j = 1}^{m} p_{j}^{n_j}
\end{align*}
La cantidad $ \binom{N}{n_1, n_2, \ldots, n_m} \equiv {N!}/{\prod_{k = 1}^{m} n_k!} $ es
llamada _coeficiente multinomial_.

Esta cantidad _no_ está implementada en `math` así que tendremos que hacerlo nosotros.
Podríamos implementarla directamente como en su definición, pero con una búsqueda rápida
en línea podemos aprender que
\begin{equation*}
  \binom{N}{{n_1}, \, {n_2}, \ldots, \, {n_m}} = \prod_{j = 1}^{m} \binom{N_j}{n_j}
\end{equation*}
con $ N_j = \sum_{i = 1}^{j} n_i $. Esta implementación requiere de menos operaciones,
por lo que podemos calcularla más rápido y con menos energía.
Puedes aprender más en @Keller2017 (¡Es un libro _open-source_!).

::: {tip} Lección de programación
No será muy común que debas implementar una función tú mismx, pero cuando debas hacerlo
investiga formas eficientes de hacerlo.

Aunque para el final de esta guía estarás convencidx de que es mejor *no implementar*
cosas tú mismx cuando ya existan librerías maduras estándar. Al trabajar en un proyecto,
no necesitas invertir tiempo en programar métodos que otras personas ya refinaron.
:::

In [ ]:
def multinom(ns: list[int]) -> int:
    "Coeficiente multinomial"

    result = 1
    nsum = 0
    for j in range(len(ns)):
        nsum += ns[j]
        result *= math.comb(nsum, ns[j])

    return result

Para verificar nuestra implementación, podemos comparar el caso binomial con el
coeficiente binomial en `math`.

In [ ]:
assert math.comb(particle_number, up_number) == multinom(
    [up_number, particle_number - up_number]
)

Usar `assert` hace a nuestro programa salir con error si la afirmación no se cumple.

Ahora sí, implementemos la función de probabilidad.

In [ ]:
def probability_multcrit(ns: list[int], ps: list[float]) -> float:
    """
    Probabilidad multi-criterio.

    Calcula la probabilidad de que ns[j] partículas cumplan el criterio j,
    donde ps[j] es la probabilidad de este criterio.
    """
    if not math.isclose(math.fsum(ps), 1.0):
        raise ValueError("Probabilidades no suman 1.")

    if not all([0 <= p <= 1 for p in ps]):
        raise ValueError("Probabilidades deben estar entre 0 y 1.")

    if len(ns) != len(ps):
        raise ValueError("Dimensiones de n y p no coinciden.")

    return multinom(ns) * math.prod([ps[j] ** ns[j] for j in range(len(ns))])

Para probar nuestra implementación, usémosla para calcular la respuesta del ejercicio
anterior.

In [ ]:
assert math.isclose(
    probability_onecrit(particle_number, up_number, up_probability),
    probability_multcrit([up_number, particle_number - up_number], [0.5, 0.5]),
)

::: {warning} ¡Cuidado al comparar `floats`!
Debido a la aritmética de punto flotante es muy probable que calcular la misma cantidad
con dos métodos diferentes no resulte en _exactamente_ el mismo número en la memoria de
la computadora. Esto es normal y esperado, la diferencia es muy pequeña y no causa problemas
en la mayoría de contextos. Sin embargo, usar el operador `==` resultará en `False` en
estos casos.

Mejor usar `math.isclose` en estos casos.
:::

In [ ]:
help(math.isclose)

In [ ]:
print(
    "La probabilidad de que la i-ésima cara aparezca i veces es",
    f"{probability_multcrit(
      [1, 2, 3, 4, 5, 6],
      [1/6 for _ in range(6)]
      ):.3e}.",
)

> ¿Y para dados D20 (de 20 caras)? 

In [ ]:
print(f"En este caso se necesitan {sum(range(1, 21))} dados.")

In [ ]:
print(f"La probabilidad es de {probability_multcrit(
  list(range(1, 21)),
  [1/20 for _ in range(20)]
):.3e}.")

Que conveniente que, al haber definido la función, podemos reutilizar la fórmula fácilmente.